<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week04_Astar_CA1_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 4 Practical (PH):** Heuristic search — greedy, A*, admissibility and the trade-offs (CA1)

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO4, LO5

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — One loop, four algorithms | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part B — Heuristics for the 8-puzzle | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part C — Compare algorithms statistically (this is the core of CA1) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part D — Design your own heuristic (CA1 core task) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| Part E — A\* on grid maps (path planning for robots, Week 10) | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | implement **greedy best-first, A\* and weighted A\*** as one best-first loop with different priorities | Apply | LO4 |
| 2 | design and **prove / test admissibility and consistency** of heuristics | Evaluate | LO4 |
| 3 | quantify the trade-off among **completeness, optimality, time and space** | Analyse / Evaluate | LO4 |
| 4 | compare algorithms statistically (mean ± sd, 95% CI, paired Wilcoxon) | Evaluate | LO4, LO5 |

**Formal definitions.** Best-first search expands the frontier node with the smallest evaluation $f(n)$:

| Algorithm | $f(n)$ | Optimal? |
|---|---|---|
| Uniform-cost | $g(n)$ | yes |
| Greedy best-first | $h(n)$ | no |
| **A\*** | $g(n)+h(n)$ | yes if $h$ admissible (tree) / consistent (graph) |
| Weighted A\* | $g(n)+w\,h(n)$, $w\ge 1$ | cost $\le w\,C^{*}$ |

* **Admissible:** $0 \le h(n) \le h^{*}(n)$ for every $n$ ($h^{*}$ = true cost-to-go).
* **Consistent:** $h(n) \le c(n,a,n') + h(n')$ for every edge; consistent $\Rightarrow$ admissible, and $f$ never decreases along a path.
* **Dominance:** if $h_2(n)\ge h_1(n)$ for all $n$ (both admissible), A\* with $h_2$ never expands more nodes than with $h_1$.

**Do Now (3 min):** In Nigeria, you plan a road trip Lagos → Abuja. Name one *admissible* heuristic and one that is *not*. (Hint: straight-line distance vs "distance ÷ 120 km/h" when speed limits vary.)

In [ ]:
#@title 🗺️ Problem interface + Romania road map (Russell & Norvig) — shared by Weeks 3–4
# heapq = priority queue (used by UCS/A*); itertools gives us counters for fresh names
import heapq, itertools
# deque = double-ended queue; popleft() makes a fast first-in-first-out queue (BFS)
from collections import deque
# Positive infinity: a cost that is larger than any real cost
INF = float("inf")

class Problem:
    """A search problem = initial state + successor function + goal test (+ step costs)."""
    # Every search problem must say where we start; subclasses override this
    def initial(self):            raise NotImplementedError
    # ...and how to test whether a state is the goal
    def is_goal(self, s):         raise NotImplementedError
    # ...and which (action, next_state, step_cost) moves are legal from a state
    def successors(self, s):      raise NotImplementedError      # → list of (action, next_state, step_cost)

# A problem defined by a weighted graph (e.g. a road map)
class GraphProblem(Problem):
    # Constructor: edges is a list of (city_a, city_b, distance)
    def __init__(self, edges, start, goal):
        # Adjacency dictionary: city -> list of (action, neighbour, cost)
        self.adj = {}
        # Unpack each road into its two ends a, b and its length d
        for a, b, d in edges:
            # Add the move a -> b (setdefault creates the list the first time we see a)
            self.adj.setdefault(a, []).append((f"go {b}", b, d))
            # Roads are two-way, so also add the move b -> a
            self.adj.setdefault(b, []).append((f"go {a}", a, d))
        # Remember the start and goal cities (tuple assignment: two names in one line)
        self.start, self.goal = start, goal
    # The search starts in the start city
    def initial(self):       return self.start
    # A state is the goal if it equals the goal city
    def is_goal(self, s):    return s == self.goal
    # Legal moves from s; .get returns [] for a city with no roads
    def successors(self, s): return self.adj.get(s, [])

# The Romania road map from AIMA: (city, city, kilometres) triples
ROMANIA_EDGES = [("Arad","Zerind",75),("Arad","Sibiu",140),("Arad","Timisoara",118),("Zerind","Oradea",71),
    ("Oradea","Sibiu",151),("Sibiu","Fagaras",99),("Sibiu","RimnicuVilcea",80),("Timisoara","Lugoj",111),
    ("Lugoj","Mehadia",70),("Mehadia","Drobeta",75),("Drobeta","Craiova",120),("Craiova","RimnicuVilcea",146),
    ("Craiova","Pitesti",138),("RimnicuVilcea","Pitesti",97),("Pitesti","Bucharest",101),("Fagaras","Bucharest",211)]
# Straight-line distance to Bucharest, used as the A* heuristic h(n)
SLD_TO_BUCHAREST = {"Arad":366,"Bucharest":0,"Craiova":160,"Drobeta":242,"Fagaras":176,"Lugoj":244,"Mehadia":241,
    "Oradea":380,"Pitesti":100,"RimnicuVilcea":193,"Sibiu":253,"Timisoara":329,"Zerind":374}
# Build the Romania problem: start Arad, goal Bucharest
romania = GraphProblem(ROMANIA_EDGES, "Arad", "Bucharest")

# The 8-puzzle: a 3x3 board with tiles 1-8 and one blank
class EightPuzzle(Problem):
    # Goal board read row by row; 0 stands for the blank
    GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)                       # 0 = blank
    # Store the start board as a tuple (tuples are hashable, so they can go in sets)
    def __init__(self, start): self.start = tuple(start)
    # The search starts from the given board
    def initial(self):    return self.start
    # Goal test: is the board exactly the goal arrangement?
    def is_goal(self, s): return s == self.GOAL
    # List every legal slide of the blank
    def successors(self, s):
        # i = blank position; divmod gives (row, column) on the 3x3 grid; out collects moves
        i = s.index(0); r, c = divmod(i, 3); out = []
        # Try each direction: name and the row/column change (dr, dc)
        for name, dr, dc in (("Up", -1, 0), ("Down", 1, 0), ("Left", 0, -1), ("Right", 0, 1)):
            # Where the blank would land after the move
            nr, nc = r + dr, c + dc
            # Only allow the move if it stays on the board
            if 0 <= nr < 3 and 0 <= nc < 3:
                # j = index of that cell; swap blank and neighbour on a list copy
                j = nr * 3 + nc; l = list(s); l[i], l[j] = l[j], l[i]
                # Record (action name, new board as tuple, cost 1)
                out.append((name, tuple(l), 1))
        # Hand back all legal moves
        return out

def scramble(k, rng):
    """Random walk of k moves away from the goal (never undoing the previous move) → always solvable."""
    # Start at the goal and scramble it (guarantees the puzzle is solvable)
    s = EightPuzzle.GOAL; prev = None; p = EightPuzzle(s)
    # Repeat the random move k times
    for _ in range(k):
        # All successor boards except the one that undoes the previous move
        opts = [t for (_, t, _) in p.successors(s) if t != prev]
        # Remember where we were; pick a random successor using the seeded generator
        prev, s = s, opts[rng.integers(0, len(opts))]
    # Return the scrambled board
    return s

## Part A — One loop, four algorithms
The only difference between UCS, greedy and A\* is the **priority** `f_value`. Implement it, then run all four on the Romania map. Greedy follows the straight-line distance (SLD) to Bucharest; watch it choose a *cheaper-looking but costlier* route.

In [ ]:
# TODO(Student): the priority function
def f_value(mode, g, h, w=1.0):
    """mode ∈ {'ucs','greedy','astar','wastar'} → priority (smaller = expanded first)."""
    # Placeholder: returns 0 until you write the priority rule
    return 0

### 🔍 Code walkthrough — one function, four algorithms
Every algorithm in this lab is the same loop with a different **priority** $f(n)$:

| mode | $f(n)$ | what it does |
|---|---|---|
| `ucs` | $g(n)$ | cheapest path so far |
| `greedy` | $h(n)$ | looks closest to the goal |
| `astar` | $g(n)+h(n)$ | cheapest *total* estimate |
| `wastar` | $g(n)+w\,h(n)$ | trades optimality for speed |

The heap stores `(f, tie, g, state)`. The `tie` counter exists only so Python never has to compare two states. The line `if g > best_g.get(s, INF): continue` throws away *stale* entries: we found a cheaper route to that state after it was queued.

In [ ]:
#@title 🧭 Generic best-first search (re-opens a state if a cheaper path is found)
# One best-first search that covers UCS, greedy, A* and weighted A*
def best_first(problem, h, mode="astar", w=1.0):
    # Start state and a tie-breaker so the heap never compares states
    s0 = problem.initial(); tie = itertools.count()
    # Frontier entries are (f, tie, g, state)
    frontier = [(f_value(mode, 0, h(s0), w), next(tie), 0, s0)]
    # best_g = cheapest known cost to each state; parent = links to rebuild the path
    best_g = {s0: 0}; parent = {s0: None}
    # Counters for the statistics
    expanded, generated, max_frontier = 0, 1, 1
    # Loop until the frontier is empty
    while frontier:
        # Pop the entry with the smallest f
        _, _, g, s = heapq.heappop(frontier)
        # Ignore outdated entries (a cheaper route was found later)
        if g > best_g.get(s, INF): continue                      # stale entry
        # Count the expansion
        expanded += 1
        # Goal test when the state is expanded (needed for optimality)
        if problem.is_goal(s):
            # Prepare to rebuild the path
            path = []; node = s
            # Follow parent links back to the start
            while node is not None: path.append(node); node = parent[node]
            # Return path, cost and statistics
            return {"path": path[::-1], "cost": g, "expanded": expanded, "generated": generated, "max_frontier": max_frontier}
        # Look at each successor (action, state, step cost)
        for _, n, c in problem.successors(s):
            # Cost of reaching it through the current state
            ng = g + c
            # Keep it only if it is cheaper than any route found so far
            if ng < best_g.get(n, INF):
                # Record cost, parent and count the node
                best_g[n] = ng; parent[n] = s; generated += 1
                # Push with its new priority
                heapq.heappush(frontier, (f_value(mode, ng, h(n), w), next(tie), ng, n))
        # Track the maximum frontier size (memory)
        max_frontier = max(max_frontier, len(frontier))
    # No path exists
    return None

# Straight-line-distance heuristic: look up the city
h_sld = lambda city: SLD_TO_BUCHAREST[city]
# Results dictionary
rows = {}
# Run three strategies on the Romania map
for mode in ["ucs", "greedy", "astar"]:
    # Search and store the result
    r = best_first(romania, h_sld, mode); rows[mode] = r
    # Print cost, nodes expanded and the path
    print(f"{mode:<7} cost={r['cost']:>4}  expanded={r['expanded']:>2}  path={' → '.join(r['path'])}")
check("A* finds the optimal route (418)",           lambda: rows["astar"]["cost"] == 418)
check("greedy is faster but NOT optimal (450)",      lambda: rows["greedy"]["cost"] == 450 and rows["greedy"]["expanded"] < rows["astar"]["expanded"])
check("A* expands fewer nodes than UCS",             lambda: rows["astar"]["expanded"] < rows["ucs"]["expanded"])

## Part B — Heuristics for the 8-puzzle
Two classic admissible heuristics (blank excluded):

* $h_1(s)$ = number of **misplaced tiles**.
* $h_2(s)$ = sum of the tiles' **Manhattan distances** to their goal squares.

*Why admissible?* Each move shifts exactly one tile by one square, so a solution needs at least one move per misplaced tile ($h_1$) and at least as many moves as the total Manhattan distance ($h_2$). They are **relaxations**: $h_1$ solves a puzzle where a tile can jump anywhere; $h_2$ one where a tile can slide onto an occupied square.

In [ ]:
# TODO(Student): implement the two heuristics for the 8-puzzle (goal = (1,2,3,4,5,6,7,8,0))
# Goal arrangement of the 8-puzzle
GOAL = EightPuzzle.GOAL
# TODO: heuristic h1 = number of misplaced tiles
def misplaced(s):
    # Placeholder: returns 0 until you write the priority rule
    return 0

# TODO: heuristic h2 = Manhattan distance
def manhattan(s):
    # Placeholder: returns 0 until you write the priority rule
    return 0

In [ ]:
#@title ✅ Auto-check for Part B (admissibility is *tested*, not assumed)
# A sample board to test your heuristics on
S = (7, 2, 4, 5, 0, 6, 8, 3, 1)
check("h1(S) = 6 (misplaced tiles)",   lambda: misplaced(S) == 6)
check("h2(S) = 14 (Manhattan)",        lambda: manhattan(S) == 14)
check("h(goal) = 0 for both",          lambda: misplaced(EightPuzzle.GOAL) == 0 and manhattan(EightPuzzle.GOAL) == 0)

def is_admissible(h, n_inst=40, k=14):
    """h(s_i) ≤ true cost-to-go (= C* − i) for every state s_i on an optimal path."""
    # Test on several scrambled boards
    for i in range(n_inst):
        # Generate a seeded scramble of k moves
        s = scramble(k, np.random.default_rng(SEED + i))
        # Solve optimally with UCS to know the true cost
        r = best_first(EightPuzzle(s), h, "ucs")
        # Walk along the optimal path; h must never exceed the remaining true cost
        for j, st in enumerate(r["path"]):
            # Overestimate found: not admissible
            if h(st) > r["cost"] - j: return False
    # Never overestimated: admissible on these samples
    return True

# Consistency: h(s) <= cost(s,n) + h(n) for every move
def is_consistent(h, n_states=300):
    # Environment object to list successors
    p = EightPuzzle(EightPuzzle.GOAL)
    # Sample many states
    for i in range(n_states):
        # Random scramble depth between 1 and 29
        s = scramble(int(np.random.default_rng(i).integers(1, 30)), np.random.default_rng(10_000 + i))
        # Any violation means not consistent
        if any(h(s) > c + h(n) for _, n, c in p.successors(s)): return False
    # Never overestimated: admissible on these samples
    return True

check("h1 is admissible (40 instances)",  lambda: is_admissible(misplaced))
check("h2 is admissible (40 instances)",  lambda: is_admissible(manhattan))
check("h2 is consistent",                 lambda: is_consistent(manhattan))
check("h2 dominates h1 on 500 random states", lambda: all(manhattan(scramble(20, np.random.default_rng(i))) >= misplaced(scramble(20, np.random.default_rng(i))) for i in range(500)))

## Part C — Compare algorithms statistically (this is the core of CA1)
Same 20 start states for every algorithm (a **paired design**). We record nodes **expanded** (time proxy), **max frontier** (space proxy) and **path cost** (optimality).

In [ ]:
#@title 🧪 Experiment: 20 random 8-puzzles (scramble length 18)
# Experiment size: 20 puzzles, scrambled 18 moves each
N_INST, K = 20, 18
# Fixed (seeded) set of puzzles shared by all algorithms
instances = [scramble(K, np.random.default_rng(SEED + 500 + i)) for i in range(N_INST)]
# The algorithm configurations to compare
configs = {                                   # name → (heuristic, mode, w)
    "UCS (h=0)":          (lambda s: 0, "ucs", 1.0),
    "A* h1 misplaced":    (misplaced, "astar", 1.0),
    "A* h2 Manhattan":    (manhattan, "astar", 1.0),
    "Weighted A* w=2":    (manhattan, "wastar", 2.0),
    "Greedy h2":          (manhattan, "greedy", 1.0),
}
# One empty result list per configuration
res = {name: [] for name in configs}
# Start the timer
t0 = time.time()
# For every puzzle ...
for s in instances:
    # ... and every configuration ...
    for name, (h, mode, w) in configs.items():
        # ... run it and store the result
        res[name].append(best_first(EightPuzzle(s), h, mode, w))
# Log how long the experiment took
log.info("experiment finished in %.1f s", time.time() - t0)

# Optimal costs from UCS: the reference
opt = np.array([r["cost"] for r in res["UCS (h=0)"]])                 # UCS is optimal ⇒ reference costs
# Print the table header
print(f"{'algorithm':<20}{'expanded (mean ± sd)':>24}{'95% CI':>20}{'max frontier':>14}{'cost / optimal':>16}")
# Table of results
table = {}
# One row per algorithm
for name, rs in res.items():
    # Nodes expanded and mean max frontier
    ex = np.array([r["expanded"] for r in rs]); mf = np.mean([r["max_frontier"] for r in rs])
    # Cost ratio to optimal and a 95% CI of expansions
    ratio = np.mean(np.array([r["cost"] for r in rs]) / opt); m, s_, lo, hi = mean_ci(ex)
    # Store the numbers for later tests
    table[name] = dict(expanded=ex, ratio=ratio, frontier=mf)
    # Print mean ± sd, CI, frontier and cost ratio
    print(f"{name:<20}{m:>14.0f} ± {s_:<7.0f}{f'[{lo:.0f}, {hi:.0f}]':>20}{mf:>14.0f}{ratio:>16.3f}")

# Expansions per algorithm for the paired tests
ex = {n: table[n]["expanded"] for n in table}
# Wilcoxon: Manhattan A* vs misplaced-tiles A*
print("\nPaired Wilcoxon  A*(h2) vs A*(h1): p =", f"{paired_wilcoxon(ex['A* h2 Manhattan'], ex['A* h1 misplaced']):.2e}")
# Wilcoxon: misplaced-tiles A* vs UCS
print("Paired Wilcoxon  A*(h1) vs UCS   : p =", f"{paired_wilcoxon(ex['A* h1 misplaced'], ex['UCS (h=0)']):.2e}")
check("all A* variants with admissible h are optimal (ratio = 1)", lambda: table["A* h1 misplaced"]["ratio"] == 1.0 and table["A* h2 Manhattan"]["ratio"] == 1.0)
check("h2 expands fewer nodes than h1 (p < 0.001)", lambda: ex["A* h2 Manhattan"].mean() < ex["A* h1 misplaced"].mean() and paired_wilcoxon(ex["A* h2 Manhattan"], ex["A* h1 misplaced"]) < 1e-3)
check("weighted A* is within factor w=2 of optimal", lambda: table["Weighted A* w=2"]["ratio"] <= 2.0)

### Reading the table: the four-way trade-off (CCMAS LO4)
| Property | UCS | A\* ($h_2$) | Weighted A\* | Greedy |
|---|---|---|---|---|
| Complete | yes | yes | yes | **no** in general (can loop in tree search / infinite spaces) |
| Optimal | yes | yes (admissible $h$) | no, $\le w\,C^{*}$ | no |
| Time | $O(b^{d})$ | $O(b^{\Delta})$, $\Delta=h^{*}-h$ (relative error) | smaller | smallest |
| Space | $O(b^{d})$ | **still exponential** — memory is A\*'s real bottleneck (fix: IDA\*) | smaller | smallest |

**Discuss.** Which row of the table does your *measured* output support? Which quantity did you *not* measure (hint: wall-clock time and completeness)?

## Part D — Design your own heuristic (CA1 core task)
**Linear conflict** (Hansson, Mayer & Yung, 1992): if two tiles are in their goal *row* but in the wrong order, one of them must leave the row and come back, costing **2 extra moves** that Manhattan distance does not count. For each row (and column), with $k$ such tiles whose goal columns (rows) in current order form a longest increasing subsequence of length $\ell$, add $2(k-\ell)$.

$$h_{LC}(s) = h_2(s) + 2\sum_{\text{rows, cols}} (k - \ell).$$

Implement `h_custom` — either linear conflict or your **own** admissible idea — and let the checker test admissibility *and* dominance over $h_2$.

In [ ]:
# TODO(Student, CA1): implement h_custom(s) — admissible and at least as informed as Manhattan
# TODO: design your own heuristic (Extension)
def h_custom(s):
    # Starter: Manhattan distance; improve it but keep it admissible
    return manhattan(s)

In [ ]:
#@title ✅ Auto-check for h_custom (CA1 requirement: admissible AND dominates Manhattan)
check("h_custom is admissible (40 instances)", lambda: is_admissible(h_custom))
check("h_custom ≥ Manhattan everywhere (500 states)", lambda: all(h_custom(scramble(20, np.random.default_rng(i))) >= manhattan(scramble(20, np.random.default_rng(i))) for i in range(500)))
check("h_custom is strictly better somewhere", lambda: any(h_custom(scramble(20, np.random.default_rng(i))) > manhattan(scramble(20, np.random.default_rng(i))) for i in range(500)), "linear conflicts must add something")
# Expansions with your heuristic on all puzzles
ex_c = np.array([best_first(EightPuzzle(s), h_custom, "astar")["expanded"] for s in instances])
# Compare with Manhattan
print("A* h_custom expanded:", f"{ex_c.mean():.0f} ± {ex_c.std(ddof=1):.0f}   vs Manhattan {ex['A* h2 Manhattan'].mean():.0f}")
check("h_custom never expands more nodes on average than Manhattan", lambda: ex_c.mean() <= ex["A* h2 Manhattan"].mean())

## Part E — A\* on grid maps (path planning for robots, Week 10)
Grid with 25% random obstacles, 4-connected moves with unit cost. Compare the **Manhattan**, **Euclidean** and **zero** heuristics over 10 random maps. All three are admissible; which is most informed on a 4-connected grid?

In [ ]:
#@title 🧱 Grid map + A* (10 random maps)
# Grid-navigation problem with walls
class GridProblem(Problem):
    # Store the grid, start and goal
    def __init__(self, grid, start, goal): self.grid, self.start, self.goal = grid, start, goal
    # Search starts at the start cell
    def initial(self):    return self.start
    # Goal test
    def is_goal(self, s): return s == self.goal
    # Legal moves
    def successors(self, s):
        # Position and grid size
        x, y = s; n, m = self.grid.shape; out = []
        # Try the four directions
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            # Neighbour cell
            nx, ny = x + dx, y + dy
            # Allow the move if inside the grid and not a wall
            if 0 <= nx < n and 0 <= ny < m and self.grid[nx, ny] == 0: out.append(("move", (nx, ny), 1))
        # Return the moves
        return out

# Make a random grid that is guaranteed solvable
def random_solvable_grid(seed, size=30, density=0.25):
    # Retry until a path exists
    for attempt in range(100):                                 # resample until a path exists
        # Seeded generator
        rng = np.random.default_rng(seed + 997 * attempt)
        # Walls with probability density; start and goal are open
        grid = (rng.random((size, size)) < density).astype(int); grid[0, 0] = grid[-1, -1] = 0
        # Keep the grid if UCS finds a path
        if best_first(GridProblem(grid, (0, 0), (size-1, size-1)), lambda s: 0, "ucs"): return grid
    # Give up with a clear error
    raise RuntimeError("no solvable grid found")

# Manhattan-distance heuristic factory for a goal
manh = lambda g: (lambda s: abs(s[0]-(g[0])) + abs(s[1]-(g[1])))
# Euclidean-distance heuristic factory
eucl = lambda g: (lambda s: math.hypot(s[0]-g[0], s[1]-g[1]))
# Result lists per heuristic
grid_res = {"zero (UCS)": [], "Euclidean": [], "Manhattan": []}
# Ten random grids
grids = [random_solvable_grid(SEED + i) for i in range(10)]
# For each grid
for grid in grids:
    # Define the goal and the problem
    goal = (grid.shape[0]-1, grid.shape[1]-1); P = GridProblem(grid, (0, 0), goal)
    # Baseline: UCS
    grid_res["zero (UCS)"].append(best_first(P, lambda s: 0, "ucs")["expanded"])
    # A* with Euclidean
    grid_res["Euclidean"].append(best_first(P, eucl(goal), "astar")["expanded"])
    # A* with Manhattan
    grid_res["Manhattan"].append(best_first(P, manh(goal), "astar")["expanded"])
# Report mean, SD, CI for each
for k, v in grid_res.items(): report(k + " (expanded)", v)
# Wilcoxon test: Manhattan vs Euclidean
print("Wilcoxon Manhattan vs Euclidean: p =", f"{paired_wilcoxon(grid_res['Manhattan'], grid_res['Euclidean']):.4f}")
check("Manhattan ≤ Euclidean ≤ zero heuristic in mean expansions", lambda: np.mean(grid_res["Manhattan"]) <= np.mean(grid_res["Euclidean"]) <= np.mean(grid_res["zero (UCS)"]))

# Plotting library
import matplotlib.pyplot as plt
# Pick one grid and solve it with A*
g0 = grids[0]; goal = (29, 29); r = best_first(GridProblem(g0, (0, 0), goal), manh(goal), "astar")
# Draw the grid
plt.figure(figsize=(4.5, 4.5)); plt.imshow(g0, cmap="gray_r")
# Draw the path in red and show the cost
ys, xs = zip(*r["path"]); plt.plot(xs, ys, "r-", lw=2); plt.title(f"A* (Manhattan): cost {r['cost']}, expanded {r['expanded']}"); plt.axis("off"); plt.show()

## 📝 CA1 deliverables (10 marks) — submit this notebook + a 1–2 page PDF
| Criterion | Marks |
|---|---|
| Correct `f_value`, heuristics, and all auto-checks ✅ | 3 |
| `h_custom`: justification of admissibility (proof sketch via a *relaxed problem*) and dominance evidence | 2 |
| Experiment: ≥ 20 instances, mean ± sd, 95% CI, named test (Wilcoxon) and a sentence interpreting $p$ | 3 |
| Reflection (1–2 pages): completeness, optimality, time, space — **supported by your numbers** | 2 |

## Reflection
- Why does A\* with an inadmissible heuristic lose optimality? Give a 3-node example.
- Memory, not time, usually kills A\*. Name one algorithm that fixes this and what it gives up.

```text
Write your reflection here…
```

## Extensions
- **Types of search beyond path-finding:** implement *hill-climbing with random restarts* for the 8-queens problem and compare with A\* (local search keeps only the current state: $O(1)$ memory, but it is incomplete).
- Implement **IDA\*** (iterative-deepening A\*).
- Swap the 8-puzzle for the 15-puzzle (hint: you will need pattern databases).

In [ ]:
checks_summary()  # how many auto-checks have you passed?